# 📈 S&P 500 Market Analysis
### Sector Performance, Financial Analysis & Stock Recommendations

This notebook:
- Downloads 10 years of S&P 500 market data from Yahoo Finance
- Shows sector performance YTD, 1Y, 2Y, 3Y
- Provides a function to fetch company financials
- Generates value-based stock recommendations using fundamental screening

## 0. Install & Import Dependencies

In [ ]:
# !pip install yfinance pandas numpy matplotlib seaborn plotly requests beautifulsoup4 -q

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings
import time
from datetime import datetime, timedelta
import requests

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.2f}'.format)
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 120)

TODAY = datetime.today()
START_10Y = TODAY - timedelta(days=365 * 10)
START_OF_YEAR = datetime(TODAY.year, 1, 1)

print(f"Analysis date: {TODAY.strftime('%Y-%m-%d')}")
print(f"10-year window: {START_10Y.strftime('%Y-%m-%d')} → {TODAY.strftime('%Y-%m-%d')}")

## 1. S&P 500 Constituent Universe
Fetch the current S&P 500 list from Wikipedia and map tickers to GICS sectors.

In [ ]:
def get_sp500_constituents():
    """Scrape current S&P 500 constituents from Wikipedia."""
    url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
    tables = pd.read_html(url)
    df = tables[0][['Symbol', 'Security', 'GICS Sector', 'GICS Sub-Industry']].copy()
    df.columns = ['ticker', 'name', 'sector', 'sub_industry']
    # Fix known ticker format issues for yfinance
    df['ticker'] = df['ticker'].str.replace('.', '-', regex=False)
    return df

sp500 = get_sp500_constituents()
print(f"S&P 500 constituents loaded: {len(sp500)} companies")
print(f"\nSector breakdown:")
print(sp500['sector'].value_counts().to_string())

## 2. Download 10 Years of Market Data
We download sector ETFs (SPDR) for efficiency, plus the SPY benchmark.

In [ ]:
# SPDR Sector ETFs — proxy for each GICS sector
SECTOR_ETFS = {
    'Information Technology': 'XLK',
    'Health Care':            'XLV',
    'Financials':             'XLF',
    'Consumer Discretionary': 'XLY',
    'Communication Services': 'XLC',
    'Industrials':            'XLI',
    'Consumer Staples':       'XLP',
    'Energy':                 'XLE',
    'Utilities':              'XLU',
    'Real Estate':            'XLRE',
    'Materials':              'XLB',
}

BENCHMARK = 'SPY'
all_tickers = [BENCHMARK] + list(SECTOR_ETFS.values())

print("Downloading 10 years of daily prices for sector ETFs + SPY...")
raw = yf.download(
    all_tickers,
    start=START_10Y.strftime('%Y-%m-%d'),
    end=TODAY.strftime('%Y-%m-%d'),
    auto_adjust=True,
    progress=True
)

prices = raw['Close'].dropna(how='all')
print(f"\nPrice data shape: {prices.shape}")
print(f"Date range: {prices.index[0].date()} → {prices.index[-1].date()}")

## 3. Compute Returns Over Multiple Horizons

In [ ]:
def compute_period_return(prices: pd.DataFrame, start_date) -> pd.Series:
    """Return total return from start_date to the latest available price."""
    start = pd.Timestamp(start_date)
    # Find the first trading day on or after start_date
    available = prices.index[prices.index >= start]
    if len(available) == 0:
        return pd.Series(dtype=float)
    first = available[0]
    last  = prices.index[-1]
    return (prices.loc[last] / prices.loc[first] - 1) * 100

horizons = {
    'YTD':  START_OF_YEAR,
    '1Y':   TODAY - timedelta(days=365),
    '2Y':   TODAY - timedelta(days=365*2),
    '3Y':   TODAY - timedelta(days=365*3),
    '5Y':   TODAY - timedelta(days=365*5),
    '10Y':  TODAY - timedelta(days=365*10),
}

returns = pd.DataFrame(
    {label: compute_period_return(prices, start) for label, start in horizons.items()}
)

# Add sector labels
etf_to_sector = {v: k for k, v in SECTOR_ETFS.items()}
etf_to_sector[BENCHMARK] = 'S&P 500 (SPY)'
returns.index = returns.index.map(lambda x: etf_to_sector.get(x, x))
returns = returns.round(2)

print("Total Returns (%) by Sector and Horizon:")
display(returns.style
    .format("{:.1f}%")
    .background_gradient(cmap='RdYlGn', axis=0)
    .set_caption("Sector Performance (%) — SPDR ETFs vs SPY"))

## 4. Sector Performance Charts

In [ ]:
# ── 4a. Bar chart: YTD / 1Y / 2Y / 3Y side-by-side ──────────────────────────

focus_horizons = ['YTD', '1Y', '2Y', '3Y']
df_plot = returns[focus_horizons].drop(index='S&P 500 (SPY)', errors='ignore').sort_values('YTD')

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[f'{h} Total Return (%)' for h in focus_horizons],
    vertical_spacing=0.14,
    horizontal_spacing=0.08
)

color_scale = px.colors.diverging.RdYlGn

for i, horizon in enumerate(focus_horizons):
    row, col = divmod(i, 2)
    sorted_df = df_plot[horizon].sort_values()
    colors = ['#d73027' if v < 0 else '#1a9850' for v in sorted_df]
    fig.add_trace(
        go.Bar(
            x=sorted_df.values,
            y=sorted_df.index,
            orientation='h',
            marker_color=colors,
            text=[f'{v:.1f}%' for v in sorted_df],
            textposition='outside',
            showlegend=False,
            name=horizon
        ),
        row=row+1, col=col+1
    )

fig.update_layout(
    title=dict(text='S&P 500 Sector Performance by Time Horizon (SPDR ETFs)', font=dict(size=18)),
    height=800,
    paper_bgcolor='#0f1117',
    plot_bgcolor='#0f1117',
    font=dict(color='#e0e0e0', size=11),
)
fig.update_xaxes(showgrid=True, gridcolor='#2a2a2a', zerolinecolor='#555')
fig.update_yaxes(showgrid=False)
fig.show()

In [ ]:
# ── 4b. Indexed cumulative performance (base = 100) over 3 years ──────────────

three_years_ago = (TODAY - timedelta(days=365*3)).strftime('%Y-%m-%d')
prices_3y = prices[prices.index >= three_years_ago].copy()

# Rebase to 100
rebased = prices_3y / prices_3y.iloc[0] * 100
rebased.columns = [etf_to_sector.get(c, c) for c in rebased.columns]

fig2 = go.Figure()
palette = px.colors.qualitative.Plotly + px.colors.qualitative.Dark24

for j, col in enumerate(rebased.columns):
    lw = 3 if col == 'S&P 500 (SPY)' else 1.5
    dash = 'solid'
    fig2.add_trace(go.Scatter(
        x=rebased.index,
        y=rebased[col],
        name=col,
        line=dict(width=lw, dash=dash, color=palette[j % len(palette)]),
        hovertemplate=f'<b>{col}</b><br>%{{x|%Y-%m-%d}}<br>%{{y:.1f}}<extra></extra>'
    ))

fig2.update_layout(
    title='3-Year Cumulative Performance — Sector ETFs vs SPY (Rebased to 100)',
    xaxis_title='Date',
    yaxis_title='Growth of $100',
    height=550,
    paper_bgcolor='#0f1117',
    plot_bgcolor='#0f1117',
    font=dict(color='#e0e0e0'),
    hovermode='x unified',
    legend=dict(orientation='v', x=1.01, y=1)
)
fig2.update_xaxes(showgrid=True, gridcolor='#2a2a2a')
fig2.update_yaxes(showgrid=True, gridcolor='#2a2a2a')
fig2.show()

In [ ]:
# ── 4c. Heat map: monthly returns for SPY ────────────────────────────────────

spy_monthly = prices['SPY'].resample('ME').last().pct_change().dropna() * 100
spy_monthly.index = spy_monthly.index.to_period('M')

pivot = spy_monthly.copy()
heat_df = pd.DataFrame({
    'year':  pivot.index.year,
    'month': pivot.index.month,
    'ret':   pivot.values
})
heat_pivot = heat_df.pivot(index='year', columns='month', values='ret')
heat_pivot.columns = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

plt.figure(figsize=(16, 7))
sns.heatmap(
    heat_pivot,
    annot=True, fmt='.1f', cmap='RdYlGn', center=0,
    linewidths=0.5, linecolor='#111',
    cbar_kws={'label': 'Monthly Return (%)', 'shrink': 0.7},
    annot_kws={'size': 9}
)
plt.title('SPY Monthly Returns Heatmap (10 Years)', fontsize=14, pad=12)
plt.ylabel('Year')
plt.xlabel('')
plt.tight_layout()
plt.show()

## 5. Individual Stock Data Download

### `get_company_financials(ticker)` 
Downloads price history, balance sheet, income statement, cash flow, and key ratios for any ticker.

In [ ]:
def get_company_financials(ticker: str, verbose: bool = True) -> dict:
    """
    Download comprehensive financials for a given ticker via yfinance.

    Parameters
    ----------
    ticker : str  — e.g. 'AAPL', 'MSFT'
    verbose : bool — print a summary if True

    Returns
    -------
    dict with keys:
        info            - company metadata & key ratios
        history         - 5-year daily OHLCV DataFrame
        income_stmt     - annual income statement
        balance_sheet   - annual balance sheet
        cash_flow       - annual cash flow statement
        quarterly_income_stmt
        quarterly_balance_sheet
        quarterly_cash_flow
        recommendations - analyst recommendations
        institutional_holders
        earnings_dates  - upcoming earnings calendar
    """
    t = yf.Ticker(ticker.upper())

    data = {
        'ticker':                   ticker.upper(),
        'info':                     t.info,
        'history':                  t.history(period='5y', auto_adjust=True),
        'income_stmt':              t.income_stmt,
        'balance_sheet':            t.balance_sheet,
        'cash_flow':                t.cashflow,
        'quarterly_income_stmt':    t.quarterly_income_stmt,
        'quarterly_balance_sheet':  t.quarterly_balance_sheet,
        'quarterly_cash_flow':      t.quarterly_cashflow,
        'recommendations':          t.recommendations,
        'institutional_holders':    t.institutional_holders,
        'earnings_dates':           t.earnings_dates,
    }

    if verbose:
        info = data['info']
        print(f"{'='*60}")
        print(f"  {info.get('longName', ticker)} ({ticker.upper()})")
        print(f"  Sector: {info.get('sector', 'N/A')}  |  Industry: {info.get('industry', 'N/A')}")
        print(f"{'='*60}")
        mktcap = info.get('marketCap', 0)
        print(f"  Market Cap:        ${mktcap/1e9:.1f}B")
        print(f"  Current Price:     ${info.get('currentPrice', info.get('regularMarketPrice', 'N/A'))}")
        print(f"  52W Range:         ${info.get('fiftyTwoWeekLow','N/A')} – ${info.get('fiftyTwoWeekHigh','N/A')}")
        print(f"  P/E (TTM):         {info.get('trailingPE', 'N/A')}")
        print(f"  Forward P/E:       {info.get('forwardPE', 'N/A')}")
        print(f"  P/S (TTM):         {info.get('priceToSalesTrailing12Months', 'N/A')}")
        print(f"  P/B:               {info.get('priceToBook', 'N/A')}")
        print(f"  EV/EBITDA:         {info.get('enterpriseToEbitda', 'N/A')}")
        print(f"  PEG Ratio:         {info.get('pegRatio', 'N/A')}")
        print(f"  Dividend Yield:    {(info.get('dividendYield') or 0)*100:.2f}%")
        print(f"  Revenue (TTM):     ${info.get('totalRevenue', 0)/1e9:.1f}B")
        print(f"  Net Margin:        {(info.get('profitMargins') or 0)*100:.1f}%")
        print(f"  ROE:               {(info.get('returnOnEquity') or 0)*100:.1f}%")
        print(f"  ROA:               {(info.get('returnOnAssets') or 0)*100:.1f}%")
        print(f"  Debt/Equity:       {info.get('debtToEquity', 'N/A')}")
        print(f"  Free Cash Flow:    ${info.get('freeCashflow', 0)/1e9:.1f}B")
        print(f"  Beta:              {info.get('beta', 'N/A')}")
        print(f"  Short % Float:     {(info.get('shortPercentOfFloat') or 0)*100:.1f}%")
        rec = data['recommendations']
        if rec is not None and not rec.empty:
            latest = rec.iloc[0]
            print(f"  Analyst Action:    {latest.get('Action', 'N/A')} → {latest.get('To Grade', 'N/A')} ({latest.get('Firm','N/A')})")
        print(f"{'='*60}")

    return data


# ── Example usage ─────────────────────────────────────────────────────────────
example = get_company_financials('AAPL')
print("\nAnnual Income Statement (last 4 years):")
display(example['income_stmt'].loc[['Total Revenue','Gross Profit','Net Income']].T)

## 6. Value Screen — Which Stocks Look Cheap?

We screen the full S&P 500 universe using a multi-factor value framework:

| Factor | Weight | Rationale |
|--------|--------|-----------|
| Forward P/E vs sector median | 25% | Core earnings cheapness |
| EV/EBITDA vs sector median | 20% | Capital-structure-neutral multiple |
| P/S vs sector median | 15% | Avoids earnings manipulation |
| PEG ratio | 15% | Adjusts for growth |
| P/B vs sector median | 10% | Asset cheapness |
| Free Cash Flow Yield | 15% | Quality + value signal |

In [ ]:
def fetch_fundamentals_batch(tickers: list, delay: float = 0.05) -> pd.DataFrame:
    """
    Fetch key valuation metrics for a list of tickers.
    Returns a DataFrame with one row per ticker.
    """
    records = []
    n = len(tickers)
    for i, ticker in enumerate(tickers):
        if i % 50 == 0:
            print(f"  Progress: {i}/{n} tickers...")
        try:
            info = yf.Ticker(ticker).info
            records.append({
                'ticker':          ticker,
                'name':            info.get('shortName', ticker),
                'sector':          info.get('sector', 'Unknown'),
                'market_cap':      info.get('marketCap', np.nan),
                'forward_pe':      info.get('forwardPE', np.nan),
                'trailing_pe':     info.get('trailingPE', np.nan),
                'ev_ebitda':       info.get('enterpriseToEbitda', np.nan),
                'ps_ratio':        info.get('priceToSalesTrailing12Months', np.nan),
                'pb_ratio':        info.get('priceToBook', np.nan),
                'peg_ratio':       info.get('pegRatio', np.nan),
                'fcf':             info.get('freeCashflow', np.nan),
                'market_cap_raw':  info.get('marketCap', np.nan),
                'net_margin':      info.get('profitMargins', np.nan),
                'roe':             info.get('returnOnEquity', np.nan),
                'debt_equity':     info.get('debtToEquity', np.nan),
                'dividend_yield':  info.get('dividendYield', 0) or 0,
                'beta':            info.get('beta', np.nan),
                'price':           info.get('currentPrice', info.get('regularMarketPrice', np.nan)),
                '52w_low':         info.get('fiftyTwoWeekLow', np.nan),
                '52w_high':        info.get('fiftyTwoWeekHigh', np.nan),
            })
        except Exception:
            pass
        time.sleep(delay)

    df = pd.DataFrame(records)
    # Compute FCF yield = FCF / market cap
    df['fcf_yield'] = df['fcf'] / df['market_cap_raw']
    # % distance from 52W high (lower = more beaten down)
    df['pct_from_52w_high'] = (df['price'] / df['52w_high'] - 1) * 100
    return df


print("Fetching fundamentals for S&P 500 constituents (this takes 3–5 minutes)...")
tickers = sp500['ticker'].tolist()
fundamentals = fetch_fundamentals_batch(tickers)
print(f"\nFetched data for {len(fundamentals)} companies")
fundamentals.head(3)

In [ ]:
def compute_value_scores(df: pd.DataFrame) -> pd.DataFrame:
    """
    Score each stock on value metrics vs its sector peers.
    Lower multiple → higher score (cheaper).
    FCF yield is inverted (higher yield → higher score).
    Final composite score is 0–100 (higher = cheaper / better value).
    """
    df = df.copy()

    # ── 1. Sector-relative z-scores for valuation multiples ──
    valuation_cols = ['forward_pe', 'ev_ebitda', 'ps_ratio', 'pb_ratio', 'peg_ratio']
    yield_col = 'fcf_yield'

    for col in valuation_cols + [yield_col]:
        sector_med = df.groupby('sector')[col].transform('median')
        sector_std = df.groupby('sector')[col].transform('std').replace(0, np.nan)
        df[f'z_{col}'] = (df[col] - sector_med) / sector_std

    # ── 2. Composite: low multiples = cheap, high FCF yield = cheap ──
    # Negate multiple z-scores (low P/E → high score)
    WEIGHTS = {
        'forward_pe':  0.25,
        'ev_ebitda':   0.20,
        'ps_ratio':    0.15,
        'peg_ratio':   0.15,
        'pb_ratio':    0.10,
        'fcf_yield':   0.15,   # positive: high yield = cheap
    }

    composite = pd.Series(0.0, index=df.index)
    for col, w in WEIGHTS.items():
        z = df[f'z_{col}']
        if col == 'fcf_yield':
            composite += w * z          # higher yield → higher score
        else:
            composite += w * (-z)       # lower multiple → higher score

    # ── 3. Scale to 0–100 ──
    mn, mx = composite.min(), composite.max()
    df['value_score'] = ((composite - mn) / (mx - mn) * 100).round(1)

    # ── 4. Quality filter: exclude deeply negative ROE or very high debt ──
    df['quality_pass'] = (
        (df['roe'].fillna(0) > -0.10) &
        (df['debt_equity'].fillna(999) < 200) &
        (df['net_margin'].fillna(0) > -0.05)
    )

    return df


scored = compute_value_scores(fundamentals)
print("Value scoring complete.")

In [ ]:
# ── Top value picks: quality-filtered, min $5B market cap, positive FCF ──────

MIN_MKTCAP = 5e9   # $5B minimum

screened = scored[
    scored['quality_pass'] &
    (scored['market_cap_raw'] >= MIN_MKTCAP) &
    (scored['fcf'].fillna(0) > 0) &
    (scored['forward_pe'].notna()) &
    (scored['forward_pe'] > 0)
].copy()

top_value = screened.sort_values('value_score', ascending=False).head(25)

display_cols = [
    'ticker', 'name', 'sector',
    'price', 'pct_from_52w_high',
    'forward_pe', 'ev_ebitda', 'ps_ratio', 'pb_ratio', 'peg_ratio',
    'fcf_yield', 'net_margin', 'roe', 'dividend_yield',
    'value_score'
]

display_df = top_value[display_cols].rename(columns={
    'pct_from_52w_high': '% from 52W Hi',
    'forward_pe':        'Fwd P/E',
    'ev_ebitda':         'EV/EBITDA',
    'ps_ratio':          'P/S',
    'pb_ratio':          'P/B',
    'peg_ratio':         'PEG',
    'fcf_yield':         'FCF Yield',
    'net_margin':        'Net Margin',
    'roe':               'ROE',
    'dividend_yield':    'Div Yield',
    'value_score':       'Value Score'
})

print("\n📋 Top 25 Value Opportunities in the S&P 500")
print("(Quality-filtered: +FCF, +ROE, manageable debt, market cap > $5B)\n")

display(display_df.set_index('ticker').style
    .format({
        'price':          '${:.2f}',
        '% from 52W Hi':  '{:.1f}%',
        'Fwd P/E':        '{:.1f}x',
        'EV/EBITDA':      '{:.1f}x',
        'P/S':            '{:.1f}x',
        'P/B':            '{:.1f}x',
        'PEG':            '{:.2f}',
        'FCF Yield':      '{:.1%}',
        'Net Margin':     '{:.1%}',
        'ROE':            '{:.1%}',
        'Div Yield':      '{:.2%}',
        'Value Score':    '{:.1f}',
    })
    .background_gradient(subset=['Value Score'], cmap='YlGn')
    .background_gradient(subset=['Fwd P/E'], cmap='RdYlGn_r')
    .background_gradient(subset=['FCF Yield'], cmap='YlGn')
    .set_caption("Value Score: 0=expensive, 100=cheapest vs sector peers")
)

In [ ]:
# ── Bubble chart: Value Score vs FCF Yield, sized by market cap ──────────────

top50 = screened.sort_values('value_score', ascending=False).head(50).dropna(
    subset=['value_score', 'fcf_yield', 'forward_pe']
)

fig3 = px.scatter(
    top50,
    x='fcf_yield',
    y='forward_pe',
    size='market_cap_raw',
    color='sector',
    text='ticker',
    hover_data={'name': True, 'value_score': True, 'net_margin': ':.1%', 'roe': ':.1%'},
    size_max=50,
    title='Value Map: FCF Yield vs Forward P/E — Top 50 Value Candidates',
    labels={
        'fcf_yield':   'FCF Yield (higher = better value)',
        'forward_pe':  'Forward P/E (lower = cheaper)',
    },
    template='plotly_dark'
)

fig3.update_traces(textposition='top center', textfont=dict(size=9))
fig3.update_layout(
    height=600,
    paper_bgcolor='#0f1117',
    plot_bgcolor='#0f1117',
    font=dict(color='#e0e0e0'),
)
# Invert y-axis: lower P/E at top
fig3.update_yaxes(autorange='reversed')
fig3.show()

In [ ]:
# ── Value picks by sector ─────────────────────────────────────────────────────

top_by_sector = (
    screened
    .sort_values('value_score', ascending=False)
    .groupby('sector')
    .head(3)
    .sort_values(['sector', 'value_score'], ascending=[True, False])
)

print("\n🏆 Top 3 Value Picks Per Sector\n")
print(f"{'Sector':<28} {'Ticker':<8} {'Name':<35} {'Fwd P/E':>8} {'EV/EBT':>8} "
      f"{'FCF Yld':>8} {'ROE':>7} {'Score':>7}")
print("-" * 115)

current_sector = None
for _, row in top_by_sector.iterrows():
    sector_label = row['sector'] if row['sector'] != current_sector else ''
    current_sector = row['sector']
    fpe   = f"{row['forward_pe']:.1f}x" if pd.notna(row['forward_pe']) else 'N/A'
    eve   = f"{row['ev_ebitda']:.1f}x"  if pd.notna(row['ev_ebitda'])  else 'N/A'
    fcfy  = f"{row['fcf_yield']:.1%}"   if pd.notna(row['fcf_yield'])  else 'N/A'
    roe   = f"{row['roe']:.1%}"         if pd.notna(row['roe'])        else 'N/A'
    print(f"{sector_label:<28} {row['ticker']:<8} {row['name'][:34]:<35} "
          f"{fpe:>8} {eve:>8} {fcfy:>8} {roe:>7} {row['value_score']:>7.1f}")

## 7. Deep-Dive: Get Full Financials on a Specific Stock
Use `get_company_financials()` defined in Section 5 to pull complete data on any name.

In [ ]:
# ── Change TICKER to any S&P 500 name you want to examine ────────────────────
TICKER = 'GOOGL'   # <─── edit me

data = get_company_financials(TICKER)

# Plot 5-year price + volume
hist = data['history']

fig4 = make_subplots(rows=2, cols=1, shared_xaxes=True,
                     row_heights=[0.75, 0.25], vertical_spacing=0.04)

fig4.add_trace(go.Scatter(
    x=hist.index, y=hist['Close'],
    name='Price', line=dict(color='#00d4aa', width=1.5)
), row=1, col=1)

fig4.add_trace(go.Bar(
    x=hist.index, y=hist['Volume'],
    name='Volume', marker_color='#4488ff', opacity=0.5
), row=2, col=1)

fig4.update_layout(
    title=f"{data['info'].get('longName', TICKER)} — 5-Year Price & Volume",
    height=500,
    paper_bgcolor='#0f1117',
    plot_bgcolor='#0f1117',
    font=dict(color='#e0e0e0'),
    showlegend=False
)
fig4.update_xaxes(showgrid=True, gridcolor='#2a2a2a')
fig4.update_yaxes(showgrid=True, gridcolor='#2a2a2a')
fig4.show()

In [ ]:
# ── Annual financials summary ─────────────────────────────────────────────────
is_ = data['income_stmt']
bs_ = data['balance_sheet']
cf_ = data['cash_flow']

key_rows_is = [r for r in ['Total Revenue', 'Gross Profit', 'Operating Income',
                             'Net Income', 'EBITDA'] if r in is_.index]
key_rows_bs = [r for r in ['Total Assets', 'Total Debt', 'Stockholders Equity',
                             'Cash And Cash Equivalents'] if r in bs_.index]
key_rows_cf = [r for r in ['Operating Cash Flow', 'Free Cash Flow', 'Capital Expenditure'] if r in cf_.index]

print(f"\n── Income Statement (Annual, $M) ──")
display((is_.loc[key_rows_is] / 1e6).round(0).T.style.format('{:,.0f}'))

print(f"\n── Balance Sheet (Annual, $M) ──")
display((bs_.loc[key_rows_bs] / 1e6).round(0).T.style.format('{:,.0f}'))

print(f"\n── Cash Flow (Annual, $M) ──")
display((cf_.loc[key_rows_cf] / 1e6).round(0).T.style.format('{:,.0f}'))

## 8. Summary Dashboard

In [ ]:
# ── Final summary: best value ideas with investment thesis tags ───────────────

print("\n" + "="*70)
print("  📊 S&P 500 VALUE OPPORTUNITIES — SUMMARY")
print(f"  As of {TODAY.strftime('%B %d, %Y')}")
print("="*70)
print("""
METHODOLOGY NOTE
━━━━━━━━━━━━━━━
The Value Score ranks stocks on 6 metrics vs. sector peers:
  • Forward P/E (25%)  • EV/EBITDA (20%)  • P/S (15%)
  • PEG ratio (15%)    • P/B (10%)        • FCF Yield (15%)

Stocks passing quality filters (positive FCF, ROE > -10%,
D/E < 200%, net margin > -5%, market cap > $5B) are ranked.

⚠️  DISCLAIMER: This is a quantitative screen, not financial
advice. Always conduct your own due diligence. Past performance
and model scores do not guarantee future returns.
""")

top5 = screened.sort_values('value_score', ascending=False).head(5)
print("TOP 5 OVERALL VALUE PICKS:")
print("-"*70)
for rank, (_, row) in enumerate(top5.iterrows(), 1):
    print(f"  #{rank}  {row['ticker']:<6}  {row['name']:<35}  Score: {row['value_score']:.0f}/100")
    print(f"        {row['sector']}")
    metrics = []
    if pd.notna(row['forward_pe']):  metrics.append(f"Fwd P/E {row['forward_pe']:.1f}x")
    if pd.notna(row['ev_ebitda']):   metrics.append(f"EV/EBITDA {row['ev_ebitda']:.1f}x")
    if pd.notna(row['fcf_yield']):   metrics.append(f"FCF Yield {row['fcf_yield']:.1%}")
    if pd.notna(row['roe']):         metrics.append(f"ROE {row['roe']:.1%}")
    print(f"        {' | '.join(metrics)}")
    print()

print("="*70)
print("Run get_company_financials(ticker) for a full deep-dive on any name.")
print("="*70)